In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# Policy gradient III: a learned critic

## Where this session fits

> **The limitation we start from: REINFORCE's weight is still one noisy sampled return, and with a single end reward every step gets the same weight**

- **The question:** can a learned estimate replace part of the sampled return, so that the weight is less noisy and each step gets its own?
- **Position: Part B (policy gradient), session 3 of its 4 sessions**
    - B1 gave the theorem
    - B2 filled slot 1 with a baseline and the advantage, and gave REINFORCE, which needs no critic
    - this session fills slot 1 again, with an advantage from a **critic**, and part of slot 3: the baseline comes from the critic
- **The three slots** of the plan from B1 (theorem): the parts of the gradient formula that the methods change
    1. **slot 1, the weight** $w$ on each gradient term **← this session**
    2. **slot 2, how far** one update moves the policy
    3. **slot 3, where the baseline comes from** (inside slot 1) **← this session, in part**
- **Series goal:** choose good actions by trial and error, from small tables to robots and language models

**By the end you can**

1. say what a critic is: its input, its output, and what it is trained on
2. say the two roles of a critic: a baseline (no bias) and a bootstrap (bias)
3. say why the TD error is a sample of the advantage
4. read one actor-critic step, with its two updates
5. say what the dial between the return and the TD error trades, and place actor-critic in the slot table


**Recall: the advantage form of the gradient from B2**

> **The weight on each score function is an advantage: the return minus a baseline, or an action value minus the state value.**

$$
\nabla_\theta J(\theta) = \Exp{\traj \sim \pi_\theta} \left[ \sum_{\tt=0}^{|\traj|-1} \nabla_\theta \log \pi_\theta(\actseq_{\traj,\tt} \mid \stateseq_{\traj,\tt}) \; \advseq_{\traj,\tt} \right]
$$
- the advantage: $\advseq = \actvalfun_\pi(\state, \act) - \statevalfun_\pi(\state)$
- a baseline $b(\state)$ that ignores the action leaves the expected gradient unchanged (the zero-mean score lemma)
- this session learns $\statevalfun$ with a network, to estimate the advantage

**Where this session sits in Part B**

<img src="images/partB_arc_b3.png" width=95%>


# What a critic is

**Why this section?** The word has appeared since the last session, with only a gloss. Here is the definition, with its input, output and loss.

> **A critic is a second network, $\statevalfun_\phi$, that learns the state value: how much return to expect from a state, as in Part A.**

- the **critic** $\statevalfun_\phi$
    - input: a state $\state$
    - output: one number, the expected return from $\state$
    - parameters: $\phi$ (the policy's parameters are $\theta$)
    - trained by regression: its loss is the squared TD error, defined below
- the **actor** is the policy $\pi_\theta$
- the critic is a *use* of Part A's value machinery, not a return to value-based control
    - the policy is still $\pi_\theta$, not an argmax over values

**A critic can play two roles**

> **Both roles lower the variance. As a baseline the critic adds no bias. As a bootstrap it lowers the variance much further, and adds bias.**

- **role 1: a baseline.** $b(\state) = \statevalfun_\phi(\state)$ depends on the state, not on the action
    - the weight is the sampled return-to-go minus the critic's value
    - no bias, even if the critic is inaccurate: the baseline condition of B2 (weight) holds, by the zero-mean score lemma
    - an inaccurate baseline keeps the expected gradient but loses some of the variance reduction (a very bad one can add variance)
- **role 2: a bootstrap.** replace the rest of the sampled return by the critic's estimate
    - the weight uses one reward and the critic's value of the next state
    - lower variance, but an inaccurate critic biases the weight (the TD bias of sampling)
- **inaccurate** means $\statevalfun_\phi(\state) \neq \statevalfun_\pi(\state)$, for any reason
    - a bare $\pi$ in $\statevalfun_\pi$ means the current policy $\pi_\theta$, as in B2
    - not yet trained, or trained toward targets that are still moving
    - a network too small to fit the values
    - a policy that has changed since the critic was fitted
- actor-critic, next, uses both at once
- a naming note
    - Sutton and Barto reserve "actor-critic" for methods that bootstrap (role 2)
    - a learned baseline alone they call REINFORCE with a baseline
    - here "critic" covers both roles


# Actor-critic: the critic's TD error is the weight

**Why this section?** The third fix: replace the tail of the sampled return by a learned estimate, exactly as TD replaced the tail of the Monte Carlo return.

> **Actor-critic: the weight is the critic's one-step error, the TD error $\delta_\tt$. It is a sample of the advantage.**

$$
\delta_\tt = \rewseq_{\tt+1} + \disc \, \statevalfun_\phi(\stateseq_{\tt+1}) - \statevalfun_\phi(\stateseq_\tt)
\qquad \advseq_\tt \approx \delta_\tt
$$

- $\delta_\tt$ is the reward, plus the critic's value of the next state, minus its value of this state
- it is the critic's **surprise**: what happened minus what the critic predicted
    - what happened: the reward plus the value of the next state
    - what it predicted: the value of this state
- positive: the step went better than the critic predicted
- negative: it went worse
- the two parts of $\delta_\tt$ are the two roles of the critic
    - the **baseline**: $-\statevalfun_\phi(\stateseq_\tt)$
    - the **bootstrap**: $\rewseq_{\tt+1} + \disc \statevalfun_\phi(\stateseq_{\tt+1})$ in place of the rest of the sampled return

**The same $\delta$ does two jobs**

- in sampling, $\delta$ was the error that moved a value: $\statevalfun \leftarrow \statevalfun + \alpha \, \delta$
    - the increment was the learning rate times $\delta$
- here the same number is used twice
    - it moves the critic: $\phi \leftarrow \phi + \alpha_\phi \, \delta \, \nabla_\phi \statevalfun_\phi(\state)$
    - it weights the actor's gradient term: $\theta \leftarrow \theta + \alpha \, \delta \, \nabla_\theta \log \pi_\theta(\act \mid \state)$
- the learning rate scales the whole step; **the weight is $\delta$ itself**
- the sign agrees in both jobs
    - $\delta > 0$: the step went better than expected, so the critic's value rises and the action becomes more likely
    - $\delta < 0$: the opposite
- why $\delta$ may serve as the weight: the next slide

**Why the TD error is a sample of the advantage**

- the exact advantage is $\actvalfun_\pi(\state, \act) - \statevalfun_\pi(\state)$ (B2 weight)
- the one-step target is a sample of $\actvalfun_\pi$
    - $\E[\rewseq_{\tt+1} + \disc \, \statevalfun_\pi(\stateseq_{\tt+1}) \mid \stateseq_\tt, \actseq_\tt] = \actvalfun_\pi(\stateseq_\tt, \actseq_\tt)$
- so $\delta_\tt$ is a sample of $\actvalfun_\pi - \statevalfun_\pi$: the advantage
    - exactly so only if the critic equals $\statevalfun_\pi$
    - otherwise $\approx$, and the gap is the bias of the bootstrap


**One actor-critic step**

- sample $\act \sim \pi_\theta(\cdot \mid \state)$
- observe $\rew$ and $\state'$
- $\delta \leftarrow \rew + \disc\,\statevalfun_\phi(\state') - \statevalfun_\phi(\state)$ (the critic's surprise; $\statevalfun_\phi(\text{terminal}) = 0$)
- critic: $\phi \leftarrow \phi + \alpha_\phi\,\delta\,\nabla_\phi \statevalfun_\phi(\state)$ (reduce its error; the target $\rew + \disc\,\statevalfun_\phi(\state')$ is held constant: a *semi-gradient* step)
- actor: $\theta \leftarrow \theta + \alpha\,\delta\,\nabla_\theta \log \pi_\theta(\act \mid \state)$ (weight by the surprise)


**One actor-critic step, with numbers**

- take $\disc = 0.9$, the critic values $\statevalfun_\phi(\state) = 0.5$ and $\statevalfun_\phi(\state') = 0.6$, and a step that earned $\rew = 1$
    - $\delta = 1 + 0.9 \cdot 0.6 - 0.5 = 1.04$
    - positive: better than the critic expected
    - the actor makes $\act$ more likely; the critic raises $\statevalfun_\phi(\state)$ toward $1.54$
- another step, with $\rew = 0$, $\statevalfun_\phi(\state) = 0.8$ and $\statevalfun_\phi(\state') = 0.6$
    - $\delta = 0 + 0.9 \cdot 0.6 - 0.8 = -0.26$
    - negative: worse than the critic expected
    - the actor makes $\act$ less likely; the critic lowers $\statevalfun_\phi(\state)$ toward $0.54$


**What the critic buys and what it costs**

- **buys:** lower variance than a full sampled return
    - $\delta_\tt$ contains *one* reward and one value, not the whole tail of the episode
    - as TD(0) against Monte Carlo
- **costs:** **bias**: if the critic is wrong, $\delta_\tt$ is a biased advantage (the same bootstrapping bias as TD, defined in sampling)
- **costs:** a second network to train, and the two learn together: the critic's target $\statevalfun_\pi$ drifts as the actor changes
- still **on-policy**: the steps come from the current actor


## The estimator dial: from the return to the TD error

**Why this section?** The advantage can be sampled for $\tt$ steps and then closed with the critic. The number of steps is the same *depth* dial as in sampling.

> **A dial from the return (no bias, high variance) to the TD error (critic bias, low variance).**

<table>
<thead><tr><th nowrap><div align="left">Advantage weight $\advseq_\tt$</div></th><th nowrap><div align="left">Steps of sampled reward</div></th><th nowrap><div align="left">Bias</div></th><th nowrap><div align="left">Variance</div></th></tr></thead>
<tbody>
<tr><td nowrap><div align="left">return-to-go, no baseline</div></td><td nowrap><div align="left">to the end</div></td><td nowrap><div align="left">none</div></td><td nowrap><div align="left">highest</div></td></tr>
<tr><td nowrap><div align="left">return-to-go minus $b(\state)$</div></td><td nowrap><div align="left">to the end</div></td><td nowrap><div align="left">none</div></td><td nowrap><div align="left">lower</div></td></tr>
<tr><td nowrap><div align="left"><b>$p$-step</b>: $p$ rewards, then the critic</div></td><td nowrap><div align="left">$p$</div></td><td nowrap><div align="left">critic error, discounted by $\disc^p$</div></td><td nowrap><div align="left">grows with $p$ (more random rewards are summed)</div></td></tr>
<tr><td nowrap><div align="left">TD error $\delta_\tt$</div></td><td nowrap><div align="left">1</div></td><td nowrap><div align="left">critic error</div></td><td nowrap><div align="left">lowest</div></td></tr>
</tbody></table>

- the two ends of the table: the full return minus a baseline $b(\state)$ (a critic can supply $b$) and the TD error
- the first two rows use the full sampled return (the second subtracts a baseline, which a critic may supply and which costs no bias)
- only the $p$-step rows and the TD error bootstrap from the critic and carry its bias
- ("sampled for $p$ steps" above means $p$ rewards from the episode, then the critic)

**GAE: a dial between the TD error and the full return**

> **GAE weights the $p$-step advantages with a hyper-parameter $\lambda$ that acts as a dial.**

- there is a method, **GAE** (generalized advantage estimation), that combines all the $p$-step advantages
    - it weights them by a hyper-parameter $\lambda$, a number between $0$ and $1$
- in a formula: a discounted sum of TD errors

$$
\hat{\advseq}_\tt = \sum_{l=0}^{\infty} (\disc \lambda)^l \, \delta_{\tt+l}
$$

- $\lambda$ is the dial
    - $\lambda = 0$: only $l = 0$ remains, so $\hat{\advseq}_\tt = \delta_\tt$: TD(0) behavior (lowest variance, critic bias)
    - $\lambda = 1$: the sum is the full sampled return minus the critic's value $\statevalfun_\phi(\stateseq_\tt)$ (highest variance, no bootstrap bias)
    - in between: a mixture of the two
- the dial trades bias for variance, the same trade as in sampling
    - a good setting is found by experiment
- PPO uses GAE as its advantage (B4 step size)
- the details are in an **appendix**, `RL_B3_Appendix_GAE.ipynb`: worth reading before PPO


## The slot table: review, this session, preview

**Why this section?** B1 (theorem) previewed three slots. Here is what is filled so far, what this session adds, and what the next sessions change.

<table>
<thead><tr><th nowrap><div align="left">Method</div></th><th nowrap><div align="left">Slot 1: weight $\advseq$</div></th><th nowrap><div align="left">Slot 2: how far</div></th><th nowrap><div align="left">Slot 3: baseline source (inside slot 1)</div></th></tr></thead>
<tbody>
<tr><td colspan="4"><div align="left"><b>Review: filled so far</b></div></td></tr>
<tr><td nowrap><div align="left">raw theorem (B1 theorem)</div></td><td nowrap><div align="left">return, no baseline</div></td><td nowrap><div align="left">step size $\alpha$</div></td><td nowrap><div align="left">none</div></td></tr>
<tr><td nowrap><div align="left">REINFORCE (B2 weight)</div></td><td nowrap><div align="left">return-to-go minus baseline</div></td><td nowrap><div align="left">step size $\alpha$</div></td><td nowrap><div align="left">moving average of returns</div></td></tr>
<tr><td colspan="4"><div align="left"><b>This session</b></div></td></tr>
<tr><td nowrap><div align="left"><b>actor-critic</b></div></td><td nowrap><div align="left">TD error (or a $p$-step advantage, GAE)</div></td><td nowrap><div align="left">step size $\alpha$</div></td><td nowrap><div align="left">learned critic $\statevalfun_\phi$</div></td></tr>
<tr><td colspan="4"><div align="left"><b>Preview: what the next sessions change</b></div></td></tr>
<tr><td nowrap><div align="left">PPO (B4 step size)</div></td><td nowrap><div align="left">advantage from a critic (GAE)</div></td><td nowrap><div align="left"><b>clipped ratio</b></div></td><td nowrap><div align="left">learned critic</div></td></tr>
<tr><td nowrap><div align="left">GRPO (Part C)</div></td><td nowrap><div align="left">group-relative advantage</div></td><td nowrap><div align="left">clipped ratio</div></td><td nowrap><div align="left"><b>mean of a group of episodes</b></div></td></tr>
</tbody></table>

- in the preview rows, bold marks the cell that is new
- the preview rows are confirmed in B4 (step size) and in Part C
- "step size $\alpha$" in slot 2 is the plain gradient step
- B4 (step size) asks how far one update may safely go


# What we established, and what it sets up

- **Established**
    - a **critic** is a second network $\statevalfun_\phi$: input a state, output the expected return, trained by regression
    - it plays two roles: a baseline (no bias) and a bootstrap (bias, lower variance)
    - **actor-critic** weights each step by the TD error, a sample of the advantage
    - the number of sampled steps before the critic is a dial between the return (no bias, high variance) and the TD error (bias, low variance)
    - GAE averages all the dial settings (appendix)
- **Sets up**
    - **the limitation:** the weight is better but still noisy, so each step is noisy
        - and the whole family so far is on-policy: after one update the batch came from the old policy, so it is discarded
        - a big step on a noisy estimate can also ruin the policy
    - next session: slot 2, how far one update may move: the surrogate objective and PPO, which also uses a critic and GAE
    - then, in Part C, GRPO: a baseline from a group of episodes with no critic
